# ◈ ScenePilot AI — Master Execution Notebook

> **Full pipeline walkthrough: system setup → agents → sandbox → API → Docker deployment → end-to-end unit test**

---

## Execution Rules

| Rule | Detail |
|---|---|
| **Run cells top-to-bottom** | Every cell depends on the state produced by cells above it |
| **Shell cells (`!`)** | Run OS commands directly in the notebook kernel's working directory |
| **`%%writefile` cells** | Write the exact production source file to disk — do not modify unless intentional |
| **Python cells** | Execute in-process logic, unit tests, and validation routines |
| **Prerequisites** | Python 3.11+, Docker Desktop running, valid `.env` with `GROQ_API_KEY` / `GEMINI_API_KEY` |

---

### Section Map

```
SECTION 1  ── System Setup & Directory Foundations          (Cells  1 –  5)
SECTION 2  ── Architecture Blueprint & Core Data Models     (Cells  6 –  8)
SECTION 3  ── Multi-Agent Pipeline                          (Cells  9 – 13)
SECTION 4  ── Sandbox Validator Runtime                     (Cells 14 – 16)
SECTION 5  ── FastAPI Gateway                               (Cells 17 – 19)
SECTION 6  ── Docker Infrastructure & Deployment            (Cells 20 – 22)
SECTION 7  ── End-to-End Unit Test Suite                    (Cells 23 – 25)
```

---
## 📁 SECTION 1 — System Setup, Directory Foundations & Local Context

In [ ]:
# Cell 2 — Create the full project directory tree
# All paths are relative to the notebook's working directory (project root).
# The -p flag makes mkdir idempotent — safe to re-run.
!mkdir -p agents api core data/rules data/samples frontend/src/components frontend/src/hooks frontend/src/types sandbox prometheus

In [ ]:
%%writefile requirements.txt
# ── Web framework ─────────────────────────────────────────────────────────────
fastapi==0.111.0
uvicorn[standard]==0.29.0
python-dotenv==1.0.1
pydantic==2.7.1

# ── LangGraph / LangChain ─────────────────────────────────────────────────────
langgraph==0.1.5
langchain-core==0.2.5

# ── LLM clients ───────────────────────────────────────────────────────────────
groq==0.9.0
google-generativeai==0.7.2

# ── Embeddings + FAISS ────────────────────────────────────────────────────────
sentence-transformers==3.0.1
faiss-cpu==1.8.0

# ── Graph / validation ────────────────────────────────────────────────────────
networkx==3.3

# ── Observability ─────────────────────────────────────────────────────────────
prometheus-client==0.20.0
opentelemetry-api==1.24.0
opentelemetry-sdk==1.24.0
opentelemetry-exporter-otlp-proto-grpc==1.24.0

# ── Utilities ─────────────────────────────────────────────────────────────────
httpx==0.27.0

In [ ]:
%%writefile .env.example
# ── LLM API Keys ──────────────────────────────────────────────────────────────
GROQ_API_KEY=your_groq_api_key_here
GEMINI_API_KEY=your_gemini_api_key_here

# ── Story generation settings ─────────────────────────────────────────────────
TOKEN_BUDGET_LIMIT=10000
MAX_RETRIES=2

# ── Style Vault ───────────────────────────────────────────────────────────────
# Minimum cosine similarity score to pass style check (0.0–1.0)
STYLE_SIMILARITY_THRESHOLD=0.35

# ── Sandbox ───────────────────────────────────────────────────────────────────
# Set to "true" to use Docker sandbox (requires running Docker daemon)
SANDBOX_USE_DOCKER=false
SANDBOX_DOCKER_IMAGE=python:3.11-slim

# ── CORS ──────────────────────────────────────────────────────────────────────
CORS_ORIGINS=http://localhost:5173,http://localhost:3000

# ── OpenTelemetry — optional Jaeger/OTLP endpoint ────────────────────────────
# OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4317

In [ ]:
# Cell 5 — Virtual environment bootstrap + dependency install
#
# Choose the command block that matches your shell:
#
#   PowerShell / CMD (Windows):
#     python -m venv .venv
#     .venv\Scripts\activate.ps1        # PowerShell
#     .venv\Scripts\activate.bat        # CMD
#     copy .env.example .env
#
#   Bash / Zsh (macOS / Linux):
#     python3 -m venv .venv
#     source .venv/bin/activate
#     cp .env.example .env
#
# The cell below installs into the CURRENT notebook kernel (recommended when
# the notebook is already launched inside the venv).

!pip install -q -r requirements.txt
print("✅ Dependencies installed.")

# Verify the three critical packages loaded cleanly
import importlib, sys
for pkg in ["fastapi", "langgraph", "networkx", "faiss", "groq", "prometheus_client"]:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError as e:
        print(f"  ✗ {pkg}  ← MISSING: {e}")

---
## 🗺️ SECTION 2 — Architecture Blueprint & Core Data Model Layer

### Cell 6 — Architectural Data-Flow Diagram

```
┌─────────────────────────────────────────────────────────────────────────┐
│  BROWSER  http://localhost:5173                                          │
│  React 18 + Vite │ React Flow tree │ Recharts │ Blueprint SVG │ Emulator│
└──────────────────────────┬──────────────────────────────────────────────┘
                           │  HTTP  (nginx reverse proxy, 120s timeout)
┌──────────────────────────▼──────────────────────────────────────────────┐
│  FASTAPI  :8000                                                          │
│  POST /api/generate  →  run_pipeline()                                  │
│  POST /api/validate  →  validate_story()   (no LLM)                     │
│  POST /api/blueprint →  generate_blueprint()                            │
│  GET  /api/samples   →  demo library JSON files                          │
│  GET  /metrics       →  Prometheus exposition text                       │
└──────────────────────────┬──────────────────────────────────────────────┘
                           │
               run_pipeline(premise, genre, tone)
                           │
┌──────────────────────────▼──────────────────────────────────────────────┐
│  LANGGRAPH  StateGraph(ScenePilotState)                                  │
│                                                                          │
│  [generate] ──► [style_vault] ──► [sandbox]                             │
│                                       │                                  │
│               approved ◄──────────────┤                                  │
│                  │         retry<max ──┤──► [retry] ──► [generate]       │
│                  │         fail ───────┤──► [fail]                       │
│                  ▼                    │         │                        │
│             [compliance] ◄────────────┘◄────────┘                        │
│                  │                                                       │
│                 END                                                      │
└──────────────────────────┬──────────────────────────────────────────────┘
           ┌───────────────┼───────────────┐
           ▼               ▼               ▼
    Groq llama-3.3    StyleVault       NetworkX
    70b-versatile     (FAISS index     simple_cycles()
    + Gemini 2.5      all-MiniLM-L6)   + schema checks
    flash fallback
           │               │               │
           └───────────────┴───────────────┘
                           │
                    StoryStore (in-memory)
                    story_id → {story, audit, approved}
                           │
                  Prometheus counters (7 metrics)
                  ← scraped by Prometheus :9090
                  ← visualised in Grafana :3001
```

#### State machine routing decision table

| `approved` | `retry_count < max_retries` | Route taken |
|:---:|:---:|---|
| `True` | — | `sandbox → compliance → END` |
| `False` | `True` | `sandbox → retry → generate` (loop) |
| `False` | `False` | `sandbox → fail → compliance → END` |

In [ ]:
%%writefile agents/state.py
"""
ScenePilotState — shared TypedDict passed through every LangGraph node.
"""
from __future__ import annotations

from typing import Any, Optional
from typing_extensions import TypedDict


class ValidationResult(TypedDict):
    passed: bool
    issues: list[str]
    cycles_detected: int
    schema_errors: list[str]
    style_violations: list[str]


class AuditEntry(TypedDict):
    story_id: str
    fingerprint: str
    timestamp: str
    agent_spans: list[dict[str, Any]]
    token_spend: int
    validation: ValidationResult


class ScenePilotState(TypedDict):
    # ── Input ──────────────────────────────────────────────────────────────
    story_id: str
    premise: str
    genre: str          # thriller | fantasy | sci-fi | educational | marketing
    tone: float         # 0.0 (dark) → 1.0 (light)

    # ── Story payload ──────────────────────────────────────────────────────
    story: Optional[dict[str, Any]]       # raw JSON from StoryGeneratorAgent
    story_json: Optional[str]             # serialised string for sandbox

    # ── Validation state ───────────────────────────────────────────────────
    validation: Optional[ValidationResult]
    style_check: Optional[dict[str, Any]] # raw FAISS result

    # ── Control flow ───────────────────────────────────────────────────────
    approved: bool
    retry_count: int
    max_retries: int

    # ── Audit / telemetry ──────────────────────────────────────────────────
    audit: Optional[AuditEntry]
    agent_spans: list[dict[str, Any]]
    token_spend: int
    error: Optional[str]

In [ ]:
%%writefile core/telemetry.py
"""
Prometheus metrics + OpenTelemetry tracer setup.
7 counters/histograms — all prefixed scenepilot_*
"""
from __future__ import annotations

from prometheus_client import Counter, Histogram  # type: ignore

# ── Prometheus counters / histograms ──────────────────────────────────────────

STORIES_GENERATED = Counter(
    "scenepilot_stories_generated_total",
    "Total approved stories generated",
)

VALIDATION_DURATION = Histogram(
    "scenepilot_validation_duration_seconds",
    "Time spent in sandbox validation",
    buckets=[0.05, 0.1, 0.25, 0.5, 1.0, 2.5, 5.0],
)

LOOP_DETECTIONS = Counter(
    "scenepilot_loop_detections_total",
    "Number of cycle/loop detections in story graphs",
)

STYLE_VIOLATIONS = Counter(
    "scenepilot_style_violations_total",
    "FAISS tone/format violations detected",
)

SANDBOX_REJECTIONS = Counter(
    "scenepilot_sandbox_rejections_total",
    "Stories rejected by the sandbox validator",
)

AGENT_TOKEN_SPEND = Counter(
    "scenepilot_agent_token_spend_total",
    "Total LLM tokens spent across all agents",
)

BUDGET_HALTS = Counter(
    "scenepilot_budget_halts_total",
    "Times the token budget ceiling was hit",
)

# ── OpenTelemetry tracer (no-op if OTEL_EXPORTER_OTLP_ENDPOINT not set) ───────

def get_tracer(name: str = "scenepilot"):
    try:
        from opentelemetry import trace  # type: ignore
        return trace.get_tracer(name)
    except ImportError:
        return None


def setup_otel() -> None:
    try:
        import os
        from opentelemetry import trace  # type: ignore
        from opentelemetry.sdk.trace import TracerProvider  # type: ignore
        from opentelemetry.sdk.trace.export.otlp.proto.grpc.trace_exporter import (
            OTLPSpanExporter,
        )
        from opentelemetry.sdk.trace.export import BatchSpanProcessor  # type: ignore

        endpoint = os.environ.get("OTEL_EXPORTER_OTLP_ENDPOINT")
        if not endpoint:
            return
        provider = TracerProvider()
        provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint=endpoint)))
        trace.set_tracer_provider(provider)
    except Exception:
        pass  # OTel is optional

---
## 🤖 SECTION 3 — The Multi-Agent Pipeline

In [ ]:
%%writefile agents/story_generator.py
"""
StoryGeneratorAgent — calls Groq (primary) or Gemini (fallback) to turn a
story premise into a fully-structured branching narrative JSON.

Primary model:  Groq  llama-3.3-70b-versatile
Fallback model: Gemini gemini-2.5-flash
"""
from __future__ import annotations

import json
import os
import time
import uuid
from typing import Any

from agents.state import ScenePilotState

# ── LLM clients — lazy import so missing keys don't crash app startup ─────────

def _groq_client():
    from groq import Groq  # type: ignore
    return Groq(api_key=os.environ["GROQ_API_KEY"])


def _gemini_client():
    import google.generativeai as genai  # type: ignore
    genai.configure(api_key=os.environ["GEMINI_API_KEY"])
    return genai.GenerativeModel("gemini-2.5-flash")


# ── System prompt ─────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are ScenePilot, an expert interactive narrative designer.
Given a story premise, genre, and tone score (0=dark, 1=light), output ONLY a
valid JSON object matching this exact schema — no markdown fences, no commentary:

{
  "title": "<story title>",
  "genre": "<genre>",
  "scenes": [
    {
      "id": "scene_001",
      "text": "<scene description>",
      "tone": "<tense|hopeful|dark|neutral|playful>",
      "choices": [
        {"text": "<choice label>", "next": "<scene_id or null for ending>"}
      ]
    }
  ]
}

Rules:
- Generate 8–14 scenes minimum.
- Every non-ending scene must have 2–3 choices.
- Ending scenes have an empty choices array [].
- scene ids are scene_001 … scene_NNN (zero-padded to 3 digits).
- No circular references — choices must always advance the story.
- Tone must match the tone score: <=0.3 → dark, 0.3-0.7 → tense/neutral, >0.7 → hopeful/playful.
"""


def _build_user_prompt(premise: str, genre: str, tone: float) -> str:
    return (
        f"Premise: {premise}\n"
        f"Genre: {genre}\n"
        f"Tone score: {tone:.2f}\n\n"
        "Generate the full branching narrative JSON now."
    )


# ── LLM call wrappers ─────────────────────────────────────────────────────────

def _call_groq(premise: str, genre: str, tone: float) -> tuple[str, int]:
    client = _groq_client()
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": _build_user_prompt(premise, genre, tone)},
        ],
        temperature=0.7,
        max_tokens=4096,
    )
    content = response.choices[0].message.content or ""
    tokens  = response.usage.total_tokens if response.usage else 0
    return content, tokens


def _call_gemini(premise: str, genre: str, tone: float) -> tuple[str, int]:
    client   = _gemini_client()
    prompt   = SYSTEM_PROMPT + "\n\n" + _build_user_prompt(premise, genre, tone)
    response = client.generate_content(
        prompt,
        generation_config={"temperature": 0.7, "max_output_tokens": 4096},
    )
    content = response.text or ""
    tokens  = getattr(getattr(response, "usage_metadata", None), "total_token_count", 0) or 0
    return content, tokens


def _parse_story(raw: str) -> dict[str, Any]:
    """Strip markdown fences if present, then parse JSON."""
    text = raw.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text  = "\n".join(lines[1:-1]) if lines[-1].strip() == "```" else "\n".join(lines[1:])
    return json.loads(text)


# ── LangGraph node ────────────────────────────────────────────────────────────

def story_generator_node(state: ScenePilotState) -> ScenePilotState:
    span_start = time.time()
    error: str | None = None
    story: dict[str, Any] | None = None
    tokens = 0

    try:
        raw, tokens = _call_groq(state["premise"], state["genre"], state["tone"])
        story = _parse_story(raw)
    except Exception as groq_err:
        try:
            raw, tokens = _call_gemini(state["premise"], state["genre"], state["tone"])
            story = _parse_story(raw)
        except Exception as gemini_err:
            error = f"Both LLMs failed. Groq: {groq_err} | Gemini: {gemini_err}"

    span = {
        "agent":       "StoryGeneratorAgent",
        "duration_ms": int((time.time() - span_start) * 1000),
        "tokens":      tokens,
        "success":     story is not None,
    }

    return {
        **state,
        "story":       story,
        "story_json":  json.dumps(story) if story else None,
        "token_spend": state.get("token_spend", 0) + tokens,
        "agent_spans": [*state.get("agent_spans", []), span],
        "error":       error,
    }

In [ ]:
%%writefile agents/style_vault_agent.py
"""
StyleVaultAgent — FAISS + sentence-transformers gate.

Two checks are performed in sequence:
  1. Tone consistency  — scene.tone must be valid for the requested genre.
  2. FAISS similarity  — each scene's text is embedded and compared against
     the style-guide rules loaded from data/rules/*.txt.
     Scenes with cosine similarity < STYLE_SIMILARITY_THRESHOLD are flagged.

Both checks are non-blocking on exception — violations are appended as strings.
"""
from __future__ import annotations

import os
import time
from typing import Any

from agents.state import ScenePilotState

# Lazy-loaded FAISS vault singleton
_vault = None


def _get_vault():
    global _vault
    if _vault is None:
        from core.style_vault import StyleVault
        _vault = StyleVault()
        rules_dir = os.path.join(os.path.dirname(__file__), "..", "data", "rules")
        _vault.load_rules_dir(os.path.abspath(rules_dir))
    return _vault


# ── Tone mapping per genre ────────────────────────────────────────────────────

VALID_TONES_BY_GENRE: dict[str, set[str]] = {
    "thriller":    {"tense", "dark", "neutral"},
    "fantasy":     {"hopeful", "tense", "neutral", "dark"},
    "sci-fi":      {"tense", "neutral", "dark", "hopeful"},
    "educational": {"neutral", "hopeful", "playful"},
    "marketing":   {"hopeful", "playful", "neutral"},
}


def _check_tone_consistency(story: dict[str, Any], genre: str) -> list[str]:
    allowed    = VALID_TONES_BY_GENRE.get(genre, set())
    violations: list[str] = []
    for scene in story.get("scenes", []):
        tone = scene.get("tone", "neutral")
        if allowed and tone not in allowed:
            violations.append(
                f"Scene {scene.get('id', '?')} has tone '{tone}' "
                f"which is not suitable for '{genre}' genre."
            )
    return violations


def _check_with_faiss(story: dict[str, Any]) -> list[str]:
    vault      = _get_vault()
    violations: list[str] = []
    threshold  = float(os.environ.get("STYLE_SIMILARITY_THRESHOLD", "0.35"))

    for scene in story.get("scenes", []):
        text = scene.get("text", "")
        if not text:
            continue
        results = vault.query(text, k=1)
        if results:
            score, rule_text = results[0]
            if score < threshold:
                violations.append(
                    f"Scene {scene.get('id', '?')} may violate style guidelines "
                    f"(similarity={score:.3f}). Nearest rule: \"{rule_text[:80]}…\""
                )
    return violations


# ── LangGraph node ────────────────────────────────────────────────────────────

def style_vault_node(state: ScenePilotState) -> ScenePilotState:
    span_start = time.time()
    violations: list[str] = []

    story = state.get("story")
    if story is None:
        span = {"agent": "StyleVaultAgent", "duration_ms": 0,
                "violations": 0, "success": False}
        return {**state, "style_check": {"violations": []},
                "agent_spans": [*state.get("agent_spans", []), span]}

    try:
        violations.extend(_check_tone_consistency(story, state.get("genre", "")))
        violations.extend(_check_with_faiss(story))
    except Exception as exc:
        violations.append(f"StyleVault error (non-blocking): {exc}")

    span = {
        "agent":       "StyleVaultAgent",
        "duration_ms": int((time.time() - span_start) * 1000),
        "violations":  len(violations),
        "success":     True,
    }

    current_validation = state.get("validation") or {
        "passed": True, "issues": [], "cycles_detected": 0,
        "schema_errors": [], "style_violations": [],
    }

    return {
        **state,
        "style_check": {"violations": violations},
        "validation":  {**current_validation, "style_violations": violations},
        "agent_spans": [*state.get("agent_spans", []), span],
    }

In [ ]:
%%writefile agents/compliance_agent.py
"""
ComplianceAgent — SHA-256 fingerprints the final story and writes a
structured audit entry to the in-memory StoryStore.
This node always runs — even on failure — so the audit trail is never lost.
"""
from __future__ import annotations

import hashlib
import json
import time
from datetime import datetime, timezone

from agents.state import AuditEntry, ScenePilotState
from core.story_store import story_store


def compliance_node(state: ScenePilotState) -> ScenePilotState:
    span_start = time.time()

    story    = state.get("story")
    story_id = state.get("story_id", "unknown")

    # SHA-256 fingerprint — sort_keys ensures deterministic hash
    fingerprint = ""
    if story:
        raw         = json.dumps(story, sort_keys=True, ensure_ascii=False)
        fingerprint = hashlib.sha256(raw.encode()).hexdigest()

    audit: AuditEntry = {
        "story_id":    story_id,
        "fingerprint": fingerprint,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "agent_spans": state.get("agent_spans", []),
        "token_spend": state.get("token_spend", 0),
        "validation":  state.get("validation") or {
            "passed": False, "issues": [],
            "cycles_detected": 0, "schema_errors": [], "style_violations": [],
        },
    }

    # Persist full record to in-memory store
    story_store.save(story_id, {
        "story":    story,
        "audit":    audit,
        "approved": state.get("approved", False),
        "premise":  state.get("premise", ""),
        "genre":    state.get("genre", ""),
        "tone":     state.get("tone", 0.5),
    })

    span = {
        "agent":       "ComplianceAgent",
        "duration_ms": int((time.time() - span_start) * 1000),
        "fingerprint": fingerprint[:16] + "…",
        "success":     True,
    }

    return {
        **state,
        "audit":       audit,
        "agent_spans": [*state.get("agent_spans", []), span],
    }

In [ ]:
%%writefile agents/orchestrator.py
"""
Orchestrator — LangGraph StateGraph wiring all four agents with a
self-correction retry loop (default max 2 retries).

Graph topology:
  generate → style_vault → sandbox
                               ├─ approved  → compliance → END
                               ├─ retry<max → retry → generate  (loop)
                               └─ fail      → fail → compliance → END
"""
from __future__ import annotations

import uuid
from typing import Literal

from langgraph.graph import END, StateGraph  # type: ignore

from agents.state import ScenePilotState
from agents.story_generator import story_generator_node
from agents.style_vault_agent import style_vault_node
from agents.sandbox_validator import sandbox_validator_node
from agents.compliance_agent import compliance_node


# ── Routing ───────────────────────────────────────────────────────────────────

def _route_after_sandbox(
    state: ScenePilotState,
) -> Literal["compliance", "retry", "fail"]:
    if state.get("approved"):
        return "compliance"
    if state.get("retry_count", 0) < state.get("max_retries", 2):
        return "retry"
    return "fail"


def _increment_retry(state: ScenePilotState) -> ScenePilotState:
    """Bump retry counter and wipe previous story so generator runs fresh."""
    return {
        **state,
        "retry_count": state.get("retry_count", 0) + 1,
        "story":       None,
        "story_json":  None,
        "approved":    False,
        "validation":  None,
        "style_check": None,
    }


def _fail_node(state: ScenePilotState) -> ScenePilotState:
    """Terminal failure — still routes to compliance so audit is persisted."""
    return {
        **state,
        "approved": False,
        "error":    state.get("error") or "Max retries exceeded.",
    }


# ── Build graph ───────────────────────────────────────────────────────────────

def build_graph() -> StateGraph:
    graph = StateGraph(ScenePilotState)

    graph.add_node("generate",   story_generator_node)
    graph.add_node("style_vault", style_vault_node)
    graph.add_node("sandbox",    sandbox_validator_node)
    graph.add_node("compliance", compliance_node)
    graph.add_node("retry",      _increment_retry)
    graph.add_node("fail",       _fail_node)

    graph.set_entry_point("generate")

    graph.add_edge("generate",   "style_vault")
    graph.add_edge("style_vault", "sandbox")

    graph.add_conditional_edges(
        "sandbox",
        _route_after_sandbox,
        {"compliance": "compliance", "retry": "retry", "fail": "fail"},
    )

    graph.add_edge("retry",      "generate")   # self-correction loop
    graph.add_edge("compliance", END)
    graph.add_edge("fail",       "compliance") # audit even on failure

    return graph


# Compiled graph singleton — built once, reused across all requests
_compiled = None


def get_compiled_graph():
    global _compiled
    if _compiled is None:
        _compiled = build_graph().compile()
    return _compiled


# ── Public entry point ────────────────────────────────────────────────────────

def run_pipeline(premise: str, genre: str, tone: float) -> ScenePilotState:
    story_id = str(uuid.uuid4())
    initial: ScenePilotState = {
        "story_id":    story_id,
        "premise":     premise,
        "genre":       genre,
        "tone":        tone,
        "story":       None,
        "story_json":  None,
        "validation":  None,
        "style_check": None,
        "approved":    False,
        "retry_count": 0,
        "max_retries": 2,
        "audit":       None,
        "agent_spans": [],
        "token_spend": 0,
        "error":       None,
    }
    return get_compiled_graph().invoke(initial)

---
## 🔬 SECTION 4 — The Isolated Sandbox Runtime Validator

In [ ]:
%%writefile sandbox/validator.py
"""
Core story validation logic.

Three independent check passes run in order:
  1. _check_schema()    — required fields, valid tones, duplicate IDs
  2. _check_cycles()    — networkx simple_cycles() on the choice digraph
  3. _check_structure() — dead-end refs, orphaned scenes, min scene count

Used both in-process (imported by SandboxValidatorAgent) and as a CLI
entrypoint (piped JSON in, result JSON out) when run inside Docker.
"""
from __future__ import annotations

from typing import Any

REQUIRED_SCENE_KEYS = {"id", "text", "tone", "choices"}
VALID_TONES         = {"tense", "hopeful", "dark", "neutral", "playful"}
MIN_SCENES          = 6


def validate_story(story: dict[str, Any]) -> dict[str, Any]:
    """
    Returns:
        {
            "passed": bool,
            "issues": [str, ...],
            "cycles_detected": int,
            "schema_errors": [str, ...],
            "structural_warnings": [str, ...],
        }
    """
    schema_errors = _check_schema(story)
    cycles, cycle_issues = _check_cycles(story)
    structural    = _check_structure(story)

    issues = schema_errors + cycle_issues + structural
    passed = len(issues) == 0

    return {
        "passed":               passed,
        "issues":               issues,
        "cycles_detected":      cycles,
        "schema_errors":        schema_errors,
        "structural_warnings":  structural,
    }


# ── 1. Schema validation ──────────────────────────────────────────────────────

def _check_schema(story: dict[str, Any]) -> list[str]:
    errors: list[str] = []

    if not isinstance(story, dict):
        return ["Story must be a JSON object."]
    if "title" not in story:
        errors.append("Missing required field: 'title'.")

    scenes = story.get("scenes")
    if not scenes or not isinstance(scenes, list):
        errors.append("Missing or empty 'scenes' array.")
        return errors

    scene_ids: set[str] = set()
    for i, scene in enumerate(scenes):
        prefix = f"Scene[{i}]"
        if not isinstance(scene, dict):
            errors.append(f"{prefix} is not an object.")
            continue
        for key in REQUIRED_SCENE_KEYS:
            if key not in scene:
                errors.append(f"{prefix} missing field '{key}'.")

        sid = scene.get("id")
        if sid:
            if sid in scene_ids:
                errors.append(f"Duplicate scene id '{sid}'.")
            scene_ids.add(str(sid))

        tone = scene.get("tone", "neutral")
        if tone not in VALID_TONES:
            errors.append(
                f"{prefix} ({sid}) has invalid tone '{tone}'. "
                f"Allowed: {sorted(VALID_TONES)}."
            )

        choices = scene.get("choices", [])
        if not isinstance(choices, list):
            errors.append(f"{prefix} ({sid}) 'choices' must be an array.")
        else:
            for j, choice in enumerate(choices):
                if "text" not in choice:
                    errors.append(f"{prefix} choice[{j}] missing 'text'.")

    return errors


# ── 2. Cycle detection (NetworkX) ─────────────────────────────────────────────

def _check_cycles(story: dict[str, Any]) -> tuple[int, list[str]]:
    try:
        import networkx as nx  # type: ignore
    except ImportError:
        return 0, []  # networkx missing — skip silently

    G = nx.DiGraph()
    for scene in story.get("scenes", []):
        sid = scene.get("id")
        if not sid:
            continue
        G.add_node(sid)
        for choice in scene.get("choices", []):
            nxt = choice.get("next")
            if nxt:
                G.add_edge(sid, nxt)

    cycles = list(nx.simple_cycles(G))
    issues = [f"Cycle detected: {' → '.join(c + [c[0]])}" for c in cycles]
    return len(cycles), issues


# ── 3. Structural checks ──────────────────────────────────────────────────────

def _check_structure(story: dict[str, Any]) -> list[str]:
    issues: list[str] = []
    scenes = story.get("scenes", [])
    if not scenes:
        return issues

    if len(scenes) < MIN_SCENES:
        issues.append(
            f"Story has only {len(scenes)} scenes (minimum required: {MIN_SCENES})."
        )

    scene_ids  = {s.get("id") for s in scenes if s.get("id")}
    referenced = set()

    for scene in scenes:
        sid     = scene.get("id")
        choices = scene.get("choices", []) if isinstance(scene.get("choices"), list) else []
        for choice in choices:
            nxt = choice.get("next")
            if nxt:
                referenced.add(nxt)
                if nxt not in scene_ids:
                    issues.append(
                        f"Scene '{sid}' choice '{choice.get('text', '?')}' "
                        f"points to non-existent scene '{nxt}'."
                    )

    # BFS reachability from root (scenes[0])
    if scenes:
        root       = scenes[0].get("id")
        scene_map  = {s.get("id"): s for s in scenes if s.get("id")}
        visited: set[str] = set()
        queue = [root]
        while queue:
            nid = queue.pop()
            if nid in visited or nid not in scene_map:
                continue
            visited.add(nid)
            for choice in scene_map[nid].get("choices", []):
                nxt = choice.get("next")
                if nxt:
                    queue.append(nxt)
        for oid in sorted(scene_ids - visited):
            issues.append(f"Scene '{oid}' is unreachable from the root scene.")

    return issues


# ── CLI entrypoint — used by Docker runner ────────────────────────────────────

if __name__ == "__main__":
    import json, sys
    data   = json.loads(sys.stdin.read())
    result = validate_story(data)
    print(json.dumps(result))
    sys.exit(0 if result["passed"] else 1)

In [ ]:
%%writefile sandbox/runner.py
"""
Sandbox runner — tries Docker-isolated execution first;
falls back to in-process validation when SANDBOX_USE_DOCKER=false
or when Docker is unavailable.
"""
from __future__ import annotations

import json
import os
import subprocess
import sys
import tempfile
from typing import Any

from sandbox.validator import validate_story as _inprocess_validate

DOCKER_IMAGE = os.environ.get("SANDBOX_DOCKER_IMAGE", "python:3.11-slim")
USE_DOCKER   = os.environ.get("SANDBOX_USE_DOCKER",   "false").lower() == "true"


def run_in_docker(story_json: str) -> dict[str, Any]:
    """
    Spin up a throwaway container with --network none --memory 128m.
    Pipes story JSON to stdin of sandbox/validator.py, parses stdout.
    """
    validator_path = os.path.join(os.path.dirname(__file__), "validator.py")

    with tempfile.NamedTemporaryFile(
        suffix=".py", mode="w", delete=False, encoding="utf-8"
    ) as tmp:
        tmp.write(open(validator_path, encoding="utf-8").read())
        tmp_path = tmp.name

    try:
        result = subprocess.run(
            [
                "docker", "run", "--rm",
                "--network", "none",
                "--memory", "128m",
                "--cpus",   "0.5",
                "-i",
                "-v", f"{tmp_path}:/validator.py:ro",
                DOCKER_IMAGE,
                "python", "/validator.py",
            ],
            input=story_json,
            capture_output=True,
            text=True,
            timeout=30,
        )
        if result.returncode not in (0, 1):
            raise RuntimeError(f"Docker exited {result.returncode}: {result.stderr}")
        return json.loads(result.stdout)
    except (subprocess.TimeoutExpired, FileNotFoundError, RuntimeError) as exc:
        raise RuntimeError(f"Docker sandbox failed: {exc}") from exc
    finally:
        os.unlink(tmp_path)


def validate_story(story: dict[str, Any]) -> dict[str, Any]:
    """Public entry point for SandboxValidatorAgent."""
    if USE_DOCKER:
        try:
            return run_in_docker(json.dumps(story))
        except Exception as exc:
            print(f"[sandbox] Docker run failed ({exc}), using in-process fallback.",
                  file=sys.stderr)
    return _inprocess_validate(story)

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# System dependencies required by FAISS and sentence-transformers
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# Run as non-root user — principle of least privilege
RUN adduser --disabled-password --gecos "" appuser \
    && chown -R appuser:appuser /app
USER appuser

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]

---
## 🔌 SECTION 5 — FastAPI Web API Gateway

In [ ]:
%%writefile api/routes.py
"""
Core story API routes.

POST /api/generate   → run_pipeline()        — full 4-agent LangGraph flow
POST /api/validate   → validate_story()      — sandbox only, no LLM
POST /api/blueprint  → generate_blueprint()  — 3-D spatial transform map
GET  /api/stories    → list all stored IDs
GET  /api/stories/{id} → retrieve stored story
GET  /api/audit/{id}   → retrieve audit record
"""
from __future__ import annotations

from typing import Any, Optional

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel, Field

from agents.orchestrator import run_pipeline
from core.story_store import story_store
from core.telemetry import STYLE_VIOLATIONS

router = APIRouter(prefix="/api", tags=["stories"])


# ── Pydantic request / response models ────────────────────────────────────────

class GenerateRequest(BaseModel):
    premise: str   = Field(..., min_length=10, max_length=2000)
    genre:   str   = Field("thriller", pattern=r"^(thriller|fantasy|sci-fi|educational|marketing)$")
    tone:    float = Field(0.5, ge=0.0, le=1.0)


class GenerateResponse(BaseModel):
    story_id:    str
    approved:    bool
    title:       Optional[str]
    scenes:      Optional[list[dict[str, Any]]]
    validation:  Optional[dict[str, Any]]
    agent_spans: list[dict[str, Any]]
    token_spend: int
    error:       Optional[str]


# ── Endpoints ─────────────────────────────────────────────────────────────────

@router.post("/generate", response_model=GenerateResponse)
def generate_story(req: GenerateRequest):
    """Run the full LangGraph pipeline and return the validated story."""
    state = run_pipeline(premise=req.premise, genre=req.genre, tone=req.tone)

    sv = (state.get("validation") or {}).get("style_violations", [])
    if sv:
        STYLE_VIOLATIONS.inc(len(sv))

    story = state.get("story") or {}
    return GenerateResponse(
        story_id    = state["story_id"],
        approved    = state.get("approved", False),
        title       = story.get("title"),
        scenes      = story.get("scenes"),
        validation  = state.get("validation"),
        agent_spans = state.get("agent_spans", []),
        token_spend = state.get("token_spend", 0),
        error       = state.get("error"),
    )


@router.get("/stories/{story_id}")
def get_story(story_id: str):
    record = story_store.get(story_id)
    if not record:
        raise HTTPException(status_code=404, detail="Story not found.")
    return record


@router.get("/audit/{story_id}")
def get_audit(story_id: str):
    record = story_store.get(story_id)
    if not record or "audit" not in record:
        raise HTTPException(status_code=404, detail="Audit not found.")
    return record["audit"]


@router.get("/stories")
def list_stories():
    return {"story_ids": story_store.all_ids()}


@router.post("/blueprint")
def generate_blueprint_endpoint(body: dict):
    """Generate 3-D spatial transforms from a story JSON."""
    from core.blueprint import generate_blueprint
    story    = body.get("story", body)
    story_id = body.get("story_id", "story")
    return generate_blueprint(story, story_id=story_id)


@router.post("/validate")
def validate_story_endpoint(body: dict):
    """Run sandbox validator only — no LLM call."""
    from sandbox.validator import validate_story
    story = body.get("story", body)
    return validate_story(story)

In [ ]:
%%writefile api/main.py
"""
FastAPI application entrypoint.

Start order:
  1. setup_otel()          — wire OTel tracer if OTEL endpoint is set
  2. CORSMiddleware        — allow Vite dev server + any configured origins
  3. include_router x2     — story routes + demo samples routes
  4. GET /metrics          — Prometheus exposition (plain GET, not app.mount)
  5. GET /health           — liveness probe
"""
from __future__ import annotations

import os

from fastapi import FastAPI, Response
from fastapi.middleware.cors import CORSMiddleware
from prometheus_client import CONTENT_TYPE_LATEST, generate_latest

from core.telemetry import setup_otel
from api.routes import router as story_router
from api.samples_route import router as samples_router

setup_otel()

app = FastAPI(
    title       = "ScenePilot AI",
    version     = "1.0.0",
    description = "AI-powered branching narrative generator with quality-gate pipeline.",
)

# CORS — origins read from env, default allows Vite dev server
origins = os.environ.get(
    "CORS_ORIGINS",
    "http://localhost:5173,http://localhost:3000"
).split(",")

app.add_middleware(
    CORSMiddleware,
    allow_origins     = origins,
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)

app.include_router(story_router)
app.include_router(samples_router)


# Plain GET — avoids 307 redirect that app.mount("/metrics") causes
@app.get("/metrics", include_in_schema=False)
def metrics():
    return Response(content=generate_latest(), media_type=CONTENT_TYPE_LATEST)


@app.get("/health")
def health():
    return {"status": "ok", "service": "scenepilot-ai"}

---
## 🐳 SECTION 6 — Docker Container Infrastructure & Deployment

In [ ]:
%%writefile docker-compose.yml
services:
  # ── Backend API ────────────────────────────────────────────────────────────
  api:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    env_file:
      - .env
    environment:
      - PYTHONUNBUFFERED=1
      - SANDBOX_USE_DOCKER=false
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock
    depends_on:
      - prometheus
    restart: unless-stopped

  # ── React Frontend (nginx:alpine3.21, non-root) ────────────────────────────
  frontend:
    build:
      context: ./frontend
      dockerfile: Dockerfile
    ports:
      - "5173:80"
    depends_on:
      - api
    restart: unless-stopped

  # ── Prometheus ─────────────────────────────────────────────────────────────
  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus/prometheus.yml:/etc/prometheus/prometheus.yml:ro
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.path=/prometheus"
    restart: unless-stopped

  # ── Grafana ────────────────────────────────────────────────────────────────
  grafana:
    image: grafana/grafana:latest
    ports:
      - "3001:3000"
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=scenepilot
    volumes:
      - grafana_data:/var/lib/grafana
    depends_on:
      - prometheus
    restart: unless-stopped

volumes:
  grafana_data:

In [ ]:
# Cell 21 — Build and start all 4 services in detached mode
#
# Expected output sequence:
#   [+] Building  ... api, frontend
#   [+] Running   ... prometheus, api, frontend, grafana
#
# First build takes ~3-5 min (downloads Python 3.11-slim + pip installs).
# Subsequent builds use layer cache and take ~30s.

!docker compose up --build -d

In [ ]:
# Cell 22 — Deployment confirmation: verify all 4 containers are Up

import subprocess, time, urllib.request, json

# Wait for api container to finish startup
print("Waiting 5 s for containers to initialise...")
time.sleep(5)

# 1. Container status
print("\n── docker compose ps ──")
result = subprocess.run(
    ["docker", "compose", "ps", "--format",
     "table {{.Name}}\t{{.Status}}\t{{.Ports}}"],
    capture_output=True, text=True
)
print(result.stdout)

# 2. Health probe
print("── GET http://localhost:8000/health ──")
try:
    with urllib.request.urlopen("http://localhost:8000/health", timeout=8) as r:
        body = json.loads(r.read())
    assert body["status"] == "ok", f"Unexpected health response: {body}"
    print(f"  ✅  {body}")
except Exception as e:
    print(f"  ✗  Health check failed: {e}")

# 3. Registered routes
print("\n── Registered API routes ──")
try:
    with urllib.request.urlopen("http://localhost:8000/openapi.json", timeout=8) as r:
        spec   = json.loads(r.read())
        routes = sorted(spec["paths"].keys())
    for route in routes:
        print(f"  {route}")
except Exception as e:
    print(f"  ✗  Could not fetch OpenAPI spec: {e}")

# 4. Prometheus metrics reachable
print("\n── Prometheus custom metrics ──")
try:
    with urllib.request.urlopen("http://localhost:8000/metrics", timeout=8) as r:
        raw = r.read().decode()
    sp_metrics = [l for l in raw.splitlines() if l.startswith("scenepilot_") and not l.startswith("#")]
    for m in sp_metrics[:7]:
        print(f"  {m}")
except Exception as e:
    print(f"  ✗  Metrics endpoint failed: {e}")

---
## 🧪 SECTION 7 — End-to-End Unit Test Suite

These three cells run entirely **in-process** — no LLM keys required, no Docker, no HTTP.
They exercise the complete data path:

```
faulty JSON fixture  →  validate_story()  →  assert cycles/schema errors caught
                     →  orchestrator retry logic simulation
                     →  assert recovery on clean fixture
                     →  assert Prometheus counters incremented
```

In [ ]:
# Cell 23 — Unit Test A: Sandbox validator catches every class of defect
import sys, os
sys.path.insert(0, os.getcwd())   # ensure local package is found

from sandbox.validator import validate_story

TESTS_PASSED = 0
TESTS_FAILED = 0

def assert_test(label: str, condition: bool, detail: str = ""):
    global TESTS_PASSED, TESTS_FAILED
    if condition:
        print(f"  ✅  {label}")
        TESTS_PASSED += 1
    else:
        print(f"  ✗   {label}  {detail}")
        TESTS_FAILED += 1

print("=" * 60)
print("TEST A — sandbox/validator.py")
print("=" * 60)

# ── A1: Missing title ─────────────────────────────────────────────────────────
r = validate_story({"scenes": [{"id":"s1","text":"x","tone":"tense","choices":[]}]})
assert_test("A1 missing title detected",
            any("title" in e for e in r["schema_errors"]),
            str(r["schema_errors"]))

# ── A2: Invalid tone caught ───────────────────────────────────────────────────
r = validate_story({
    "title": "T",
    "scenes": [{"id":"s1","text":"x","tone":"INVALID","choices":[]}]
})
assert_test("A2 invalid tone detected",
            any("INVALID" in e for e in r["schema_errors"]),
            str(r["schema_errors"]))

# ── A3: Cycle detected (scene_A → scene_B → scene_A) ─────────────────────────
cyclic_story = {
    "title": "Cyclic",
    "scenes": [
        {"id":"s1","text":"Start","tone":"tense","choices":[{"text":"go","next":"s2"}]},
        {"id":"s2","text":"Mid",  "tone":"dark", "choices":[{"text":"back","next":"s1"}]},
        {"id":"s3","text":"End",  "tone":"tense","choices":[]},
        {"id":"s4","text":"E2",   "tone":"dark", "choices":[]},
        {"id":"s5","text":"E3",   "tone":"tense","choices":[]},
        {"id":"s6","text":"E4",   "tone":"tense","choices":[]},
    ]
}
r = validate_story(cyclic_story)
assert_test("A3 cycle s1→s2→s1 detected",
            r["cycles_detected"] >= 1,
            f"cycles={r['cycles_detected']}")
assert_test("A3 story marked failed",
            r["passed"] is False)

# ── A4: Dangling reference ────────────────────────────────────────────────────
dangling_story = {
    "title": "Dangling",
    "scenes": [
        {"id":"s1","text":"Start","tone":"tense","choices":[{"text":"go","next":"GHOST"}]},
        {"id":"s2","text":"E1",   "tone":"tense","choices":[]},
        {"id":"s3","text":"E2",   "tone":"tense","choices":[]},
        {"id":"s4","text":"E3",   "tone":"dark", "choices":[]},
        {"id":"s5","text":"E4",   "tone":"dark", "choices":[]},
        {"id":"s6","text":"E5",   "tone":"dark", "choices":[]},
    ]
}
r = validate_story(dangling_story)
assert_test("A4 dangling ref to 'GHOST' detected",
            any("GHOST" in w for w in r["structural_warnings"]),
            str(r["structural_warnings"]))

# ── A5: Too few scenes ────────────────────────────────────────────────────────
tiny_story = {
    "title": "Tiny",
    "scenes": [
        {"id":"s1","text":"A","tone":"tense","choices":[{"text":"go","next":"s2"}]},
        {"id":"s2","text":"B","tone":"dark", "choices":[]},
    ]
}
r = validate_story(tiny_story)
assert_test("A5 min scene count (2 < 6) rejected",
            any("minimum" in w for w in r["structural_warnings"]),
            str(r["structural_warnings"]))

# ── A6: Clean story passes ────────────────────────────────────────────────────
clean_story = {
    "title": "Good Story",
    "scenes": [
        {"id":"s1","text":"Open",   "tone":"tense",  "choices":[{"text":"A","next":"s2"},{"text":"B","next":"s3"}]},
        {"id":"s2","text":"Path A", "tone":"dark",   "choices":[{"text":"C","next":"s4"},{"text":"D","next":"s5"}]},
        {"id":"s3","text":"Path B", "tone":"hopeful","choices":[{"text":"E","next":"s6"}]},
        {"id":"s4","text":"End 1",  "tone":"dark",   "choices":[]},
        {"id":"s5","text":"End 2",  "tone":"tense",  "choices":[]},
        {"id":"s6","text":"End 3",  "tone":"hopeful","choices":[]},
    ]
}
r = validate_story(clean_story)
assert_test("A6 clean 6-scene story passes",
            r["passed"] is True,
            str(r["issues"]))
assert_test("A6 zero cycles",  r["cycles_detected"] == 0)
assert_test("A6 zero schema errors", r["schema_errors"] == [])

In [ ]:
# Cell 24 — Unit Test B: Orchestrator retry-loop simulation
#
# We simulate the orchestrator's routing logic WITHOUT calling any LLM.
# A mock state transitions through: fail → retry → retry → fail → compliance.
# We verify retry_count increments and approved remains False at max_retries.

print("=" * 60)
print("TEST B — Orchestrator retry-loop state machine simulation")
print("=" * 60)

# Import only the routing helpers — no LangGraph invocation needed
import importlib, types

# Inline the routing functions directly (mirrors agents/orchestrator.py)
def _route_after_sandbox(state):
    if state.get("approved"):
        return "compliance"
    if state.get("retry_count", 0) < state.get("max_retries", 2):
        return "retry"
    return "fail"

def _increment_retry(state):
    return {
        **state,
        "retry_count": state.get("retry_count", 0) + 1,
        "story":       None,
        "approved":    False,
        "validation":  None,
    }

# Seed a faulty state (cycles found → approved=False)
state = {
    "story_id":    "test-001",
    "premise":     "A test story",
    "genre":       "thriller",
    "tone":        0.5,
    "story":       {"title": "Broken", "scenes": []},
    "approved":    False,
    "retry_count": 0,
    "max_retries": 2,
    "agent_spans": [],
    "token_spend": 0,
    "validation":  {"passed": False, "issues": ["Cycle detected"],
                    "cycles_detected": 1, "schema_errors": [], "style_violations": []},
    "error":       None,
}

# ── Simulate the loop ─────────────────────────────────────────────────────────
routes_taken = []
for step in range(5):  # safety cap
    route = _route_after_sandbox(state)
    routes_taken.append(route)
    if route == "retry":
        state = _increment_retry(state)
    elif route == "compliance":
        break
    elif route == "fail":
        state = {**state, "approved": False, "error": "Max retries exceeded."}
        # fail always routes to compliance next
        routes_taken.append("compliance")
        break

print(f"  Route sequence: {' → '.join(routes_taken)}")

assert_test("B1 first two routes are 'retry'",
            routes_taken[:2] == ["retry", "retry"],
            str(routes_taken))
assert_test("B2 third route is 'fail'",
            routes_taken[2] == "fail",
            str(routes_taken))
assert_test("B3 final route is 'compliance' (audit always persisted)",
            routes_taken[-1] == "compliance",
            str(routes_taken))
assert_test("B4 retry_count reached max_retries (2)",
            state["retry_count"] == 2,
            f"retry_count={state['retry_count']}")
assert_test("B5 approved remains False after exhausted retries",
            state["approved"] is False)
assert_test("B6 error message set on failure",
            "Max retries" in (state.get("error") or ""),
            str(state.get("error")))

# ── Now simulate a CLEAN story passing first time ─────────────────────────────
clean_state = {**state, "approved": True, "retry_count": 0, "error": None}
route = _route_after_sandbox(clean_state)
assert_test("B7 approved=True routes directly to compliance",
            route == "compliance",
            f"got '{route}'")

In [ ]:
# Cell 25 — Unit Test C: Prometheus counter increments + story store
#
# Verifies that the 7 telemetry counters respond correctly to agent calls
# and that StoryStore's thread-safe save/get/all_ids/delete cycle works.

print("=" * 60)
print("TEST C — Telemetry counters & StoryStore")
print("=" * 60)

from core.telemetry import (
    STORIES_GENERATED, LOOP_DETECTIONS, STYLE_VIOLATIONS,
    SANDBOX_REJECTIONS, AGENT_TOKEN_SPEND, BUDGET_HALTS, VALIDATION_DURATION,
)
from core.story_store import StoryStore

# ── C1-C5: Counter increments ─────────────────────────────────────────────────
def _counter_value(counter) -> float:
    """Read current value from a prometheus_client Counter."""
    return counter._value.get()

before_gen   = _counter_value(STORIES_GENERATED)
before_loop  = _counter_value(LOOP_DETECTIONS)
before_style = _counter_value(STYLE_VIOLATIONS)
before_rej   = _counter_value(SANDBOX_REJECTIONS)
before_tok   = _counter_value(AGENT_TOKEN_SPEND)

STORIES_GENERATED.inc()
LOOP_DETECTIONS.inc(3)
STYLE_VIOLATIONS.inc(2)
SANDBOX_REJECTIONS.inc()
AGENT_TOKEN_SPEND.inc(512)

assert_test("C1 STORIES_GENERATED incremented by 1",
            _counter_value(STORIES_GENERATED) == before_gen + 1)
assert_test("C2 LOOP_DETECTIONS incremented by 3",
            _counter_value(LOOP_DETECTIONS) == before_loop + 3)
assert_test("C3 STYLE_VIOLATIONS incremented by 2",
            _counter_value(STYLE_VIOLATIONS) == before_style + 2)
assert_test("C4 SANDBOX_REJECTIONS incremented by 1",
            _counter_value(SANDBOX_REJECTIONS) == before_rej + 1)
assert_test("C5 AGENT_TOKEN_SPEND incremented by 512",
            _counter_value(AGENT_TOKEN_SPEND) == before_tok + 512)

# ── C6-C9: StoryStore thread-safe CRUD ───────────────────────────────────────
store = StoryStore()  # fresh instance

store.save("story-abc", {"title": "Test", "approved": True})
store.save("story-xyz", {"title": "Demo", "approved": False})

assert_test("C6 save + get round-trip",
            store.get("story-abc")["title"] == "Test")
assert_test("C7 all_ids returns both IDs",
            set(store.all_ids()) == {"story-abc", "story-xyz"},
            str(store.all_ids()))
assert_test("C8 get unknown ID returns None",
            store.get("MISSING") is None)

deleted = store.delete("story-abc")
assert_test("C9 delete removes record",
            deleted is True and store.get("story-abc") is None)

# ── Final scorecard ───────────────────────────────────────────────────────────
print()
print("=" * 60)
total = TESTS_PASSED + TESTS_FAILED
print(f"  RESULT: {TESTS_PASSED}/{total} tests passed", end="")
if TESTS_FAILED == 0:
    print("  ✅  ALL PASS — pipeline is structurally sound.")
else:
    print(f"  ⚠️  {TESTS_FAILED} test(s) FAILED — review output above.")
print("=" * 60)